First we take an image. This image itself is not searchable or structured.

**Image → Text**

What it produces:

extracted words/lines

OCR reads the text from image, like a human reading a receipt.


In [ ]:
# installs Poppler tools (system-level utilities).

# Why needed:
# pdf2image needs Poppler to read and convert PDFs.
# Poppler provides commands like pdfinfo and pdftoppm.

# In your pipeline:
# This enables PDF → image conversion.
!apt-get install -y poppler-utils



# installs the Python library pdf2image.

# Why needed:
# So your code can do:

# from pdf2image import convert_from_path
# images = convert_from_path(pdf_path, dpi=300)


# In your pipeline:
# This is the direct step: PDF → images (page-wise).

!pip install pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [1]:
ls

sample_data/


In [ ]:
# Refreshes the Linux package list.
# So Colab knows the latest versions of Ubuntu packages before installing anything.
!apt-get update

# installs Poppler tools + supporting data files.
!apt-get install -y poppler-utils poppler-data

# checks where pdfinfo is installed.
!which pdfinfo

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,888 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,605 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,682 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,999 kB]
Fetched 18.6 MB in 5s (3,909 kB/s)
Read

In [ ]:
# Part A — Import libraries

from pdf2image import convert_from_path
import os

# Part B — Provide PDF input

# Path to PDF file
pdf_path = "/content/Receipt-template-example.pdf"

# Part C — Create output folder based on PDF name

# Get the PDF name without extension
pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
# Directory to save images
output_dir = f"/content/{pdf_name}"
os.makedirs(output_dir, exist_ok=True)  # Create the folder if it doesn't exist

# Part D — Convert PDF pages to images

# Convert PDF to images (one image per page)
images = convert_from_path(pdf_path, dpi=300, fmt='jpeg')

# Part E — Select only specific pages

pages = [1]  # Specify the pages you want to convert e.g., [1,2,3] → first 3 pages


# Part F — Save selected pages as images + store their paths
# List to store the names of saved images as strings
saved_image_paths = []

# Save specified pages as images in the created folder
for i in pages:
    if i <= len(images):  # Check if the page number is valid
        # Save images in the created folder with incremental filenames
        image_path = f"{output_dir}/{pdf_name}_page_{i}.jpg"
        images[i - 1].save(image_path, "JPEG")  # -1 because list is 0-indexed
        print(f"Saved {image_path}")
        saved_image_paths.append(image_path)  # Append the image path as a string to the list
    else:
        print(f"Page {i} does not exist in the PDF.")

# Part G — Print final output list
# Print the list of saved image paths as strings
print("Saved Image Paths:", saved_image_paths)


Saved /content/Receipt-template-example/Receipt-template-example_page_1.jpg
Saved Image Paths: ['/content/Receipt-template-example/Receipt-template-example_page_1.jpg']


In [ ]:
# import google.generativeai as genai
# import os

# genai.configure(api_key="YOUR API Key")

# models = genai.list_models()
# for m in models:
#     print(m.name, "->", m.supported_generation_methods)

models/gemini-2.5-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-exp-image-generation -> ['generateContent', 'countTokens', 'bidiGenerateContent']
models/gemini-2.0-flash-lite-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-exp-1206 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts -> ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts -> ['cou

In [ ]:
import google.generativeai as genai
import PIL.Image
import os

# Setup API key
genai.configure(api_key="YOUR API Key")   # set env var outside code

# Choose model
model = genai.GenerativeModel("gemini-2.5-flash")


# Load image
img = PIL.Image.open("/content/Receipt-template-example/Receipt-template-example_page_1.jpg")

# Prompt
prompt = "Read the text in this image and output it as a Markdown table."
response = model.generate_content([prompt, img])

print(response.text)


```markdown
# ONLINE RECEIPT

**East Repair Inc.**
1912 Harvest Lane
New York, NY 12210

| **BILL TO** | **SHIP TO** | **RECEIPT #** | US-001 |
| :---------- | :---------- | :------------ | :----- |
| John Smith  | John Smith  | **RECEIPT DATE** | 11/02/2019 |
| 2 Court Square | 3787 Pineview Drive | **P.O.#** | 2312/2019 |
| New York, NY 12210 | Cambridge, MA 12210 | **DUE DATE** | 26/02/2019 |

<br>

## Receipt Total
# $154.06

| **QTY** | **DESCRIPTION** | **UNIT PRICE** | **AMOUNT** |
| :------ | :-------------- | :------------- | :--------- |
| 1       | Front and rear brake cables | 100.00         | 100.00     |
| 2       | New set of pedal arms | 15.00          | 30.00      |
| 3       | Labor 3hrs      | 5.00           | 15.00      |
|         | Subtotal        |                | 145.00     |
|         | Sales Tax 6.25% |                | 9.06       |

<br>

**PAYMENT INSTRUCTION**

Paypal email
receipt@gmail.com

Bank transfer
Routing (ABC): 0560120214

**John Smith**
*(Signat

In [ ]:
from google.colab import drive
drive.mount('/content/drive')